<a href="https://colab.research.google.com/github/hmmnyamminji/DL/blob/main/day04_practice1_%ED%94%BC%EB%A7%88_%ED%8C%8C%EC%9D%B4%ED%94%84%EB%9D%BC%EC%9D%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 피마 인디언 당뇨 예측
# 'CSV -> 판다스 탐색 -> 텐서 -> Dataset/DataLoader -> 학습' 전체 파이프라인

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# 셀 1. 데이터 불러오기
CSV_URL = "https://raw.githubusercontent.com/taehojo/deeplearning_4th/master/data/pima-indians-diabetes3.csv"
CSV_PATH = "pima-indians-diabetes3.csv"

def load_pima():
  import os
  if os.path.exists(CSV_PATH):
    print(f"로컬 파일 재사용: {CSV_PATH}")
    return pd.read_csv(CSV_PATH)

  try:
    df = pd.read_csv(CSV_URL)
    df.to_csv(CSV_PATH, index=False)
    print(f"다운로드 완료 -> {CSV_PATH} 저장 (다음 실행부턴 재사용)")
    return df
  except Exception as e:
    raise SystemExit(
        f"데이터 다운로드 실패: {e}\n"
    )

df = load_pima()
print("데이터 크기:", df.shape)
print(df.head())

다운로드 완료 -> pima-indians-diabetes3.csv 저장 (다음 실행부턴 재사용)
데이터 크기: (768, 9)
   pregnant  plasma  pressure  thickness  insulin   bmi  pedigree  age  \
0         6     148        72         35        0  33.6     0.627   50   
1         1      85        66         29        0  26.6     0.351   31   
2         8     183        64          0        0  23.3     0.672   32   
3         1      89        66         23       94  28.1     0.167   21   
4         0     137        40         35      168  43.1     2.288   33   

   diabetes  
0         1  
1         0  
2         1  
3         0  
4         1  


In [5]:
# 셀 2. 데이터 조사 - 모델링 전에 눈으로
print("\n당뇨 여부 분포:")
print(df["diabetes"].value_counts()) # 0: 500명, 1: 268 명 -> 불균형 확인

# 정답과의 상관관계
corr = df.corr()["diabetes"].drop("diabetes").sort_values(ascending=False)
print("\ndiabetes와의 상관계수:")
print(corr.round(3))

# plasma(혈당)가 가장 높다 -> 상식과 일치, 데이터가 말이 되는지 늘 확인


당뇨 여부 분포:
diabetes
0    500
1    268
Name: count, dtype: int64

diabetes와의 상관계수:
plasma       0.467
bmi          0.293
age          0.238
pregnant     0.222
pedigree     0.174
insulin      0.131
thickness    0.075
pressure     0.065
Name: diabetes, dtype: float64


In [11]:
# 셀 3. 텐서로 변환 + 학습/테스트 분리 + 표준화

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df.drop("diabetes", axis=1).values
y = df["diabetes"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

In [12]:
# 셀 4. Dataset - 데이터 천 개를 어떻게 꺼내는가

class PimaDatasets(Dataset):
  def __init__(self, X, y):
    self.X = X
    self.y = y

  def __len__(self): # 전체 몇 개
    return len(self.X)

  def __getitem__(self, idx): # idx 번째 하나
    return self.X[idx], self.y[idx]

train_ds = PimaDatasets(X_train_t, y_train_t)
test_ds = PimaDatasets(X_test_t, y_test_t)

print("\n전체 개수:", len(train_ds))
x0, y0 = train_ds[0]
print("0번째 샘플:", x0.shape, "정답:", y0.item())



전체 개수: 614
0번째 샘플: torch.Size([8]) 정답: 0.0


In [14]:
# 셀 5. DataLoader - 데이터를 몇 개씩, 어떤 순서로 꺼내는가

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

xb, yb = next(iter(train_loader))
print("\n배치 하나의 모양:", xb.shape, yb.shape)
print(f"배치 개수: {len(train_loader)}")



배치 하나의 모양: torch.Size([32, 8]) torch.Size([32, 1])
배치 개수: 20


In [17]:
# 셀 6. 모델 설계 (교재 10장)

class PimaNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
      nn.Linear(8, 16),           # 입력층 -> 은닝 1
      nn.ReLU(),                  # 활성화 함수
      nn.Linear(16, 8),           # 은닉 2
      nn.ReLU(),                  # 활성화 함수
      nn.Linear(8, 1),            # 출력층:
      nn.Sigmoid(),
    )

  def forward(self, x):
    return self.net(x)

model = PimaNet().to(device)
print("\n", model)



 PimaNet(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=8, bias=True)
    (3): ReLU()
    (4): Linear(in_features=8, out_features=1, bias=True)
    (5): Sigmoid()
  )
)


In [20]:
# 셀 7. 배치 학습 루프

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 100
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for xb, yb in train_loader: # 배치 단위로
        xb, yb = xb.to(device), yb.to(device)

        pred = model(xb)                # 예측 32명씩
        loss = loss_fn(pred, yb)        # 손실

        optimizer.zero_grad()           # 매 배치(batch)마다 기울기를 초기화하고 역전파를 수행
        loss.backward()                 # 역전파
        optimizer.step()                # 갱신

        epoch_loss += loss.item() * len(xb) # 이 배치에 해당하는 총 손실

    if (epoch + 1) % 20 == 0:
      print(f"epoch {epoch+1:3d} | 평균 loss {epoch_loss / len(train_ds):.4f}") # 전체 훈련 데이터셋에 대한 평균 손실을 계산



epoch  20 | 평균 loss 0.4379
epoch  40 | 평균 loss 0.4259
epoch  60 | 평균 loss 0.4178
epoch  80 | 평균 loss 0.4098
epoch 100 | 평균 loss 0.4029


In [21]:
# 셀 8. 평가
model.eval() # 평가 모드
correct = 0

with torch.no_grad(): # 기울기 계산을 비활성화
  for xb, yb in test_loader:
      xb, yb = xb.to(device), yb.to(device)
      pred = model(xb)
      correct += ((pred > 0.5) == yb.bool()).sum().item() # 예측값과 실제값이 같으면 True(맞은 예측)

print(f"\n 테스트 정확도: {correct / len(test_ds):.4f} ({correct}/{len(test_ds)})")

# 이 데이터는 정확도 70%~78% 사이가 정상 범위
# 무조건 정상 이라고만 찍어도 65%인 불균형 데이터
# 따라서 65%를 얼마나 넘겼는지가 중요




 테스트 정확도: 0.7468 (115/154)
